# Import

In [ ]:
import asyncio
import os
import base64
import json
from typing import Literal, Tuple, Annotated

from pydantic import BaseModel, Field
from pydantic import BaseModel, Field, ConfigDict
from math import ceil, floor

from io import BytesIO
from PIL import Image, ImageDraw, ImageFont
from PIL import ImageColor

from transformers import Qwen3VLMoeForConditionalGeneration, Qwen3VLForConditionalGeneration, AutoProcessor

from agentscope.agent import ReActAgent, UserAgent
from agentscope.formatter import OpenAIChatFormatter, OpenAIMultiAgentFormatter
from agentscope.memory import InMemoryMemory
from agentscope.model import OpenAIChatModel
from agentscope.agent import ReActAgent
from agentscope.tool import ToolResponse, Toolkit, execute_python_code, write_text_file
from agentscope.message import (
    Msg,
    Base64Source,
    TextBlock,
    URLSource,
    ThinkingBlock,
    ImageBlock,
    ToolUseBlock,
    ToolResultBlock,
)
import requests
import openai
import openslide
import uuid
import numpy as np
import cv2
import matplotlib.pyplot as plt
from qwen_vl_utils import process_vision_info, extract_vision_info

In [ ]:
def image2base(img_url):
    if os.path.exists(img_url):
        with open(img_url, "rb") as image_file:
            base64_image = base64.b64encode(image_file.read()).decode("utf-8")
    elif img_url.startswith("http://") or img_url.startswith("https://"):
        response = requests.get(img_url)
        response.raise_for_status()
        base64_image = base64.b64encode(response.content).decode("utf-8")
    else:
        raise ValueError("Invalid image URL")

    return base64_image

In [ ]:
img_url = "/data/home/zhangchen/project/RL/SlideReasoner/slidereasoner/reasource/qwen3vl_assets/spatial_understanding/dining_table.png"
img = Image.open(BytesIO(base64.b64decode(image2base(img_url))))
display(img)

In [ ]:
image2base(img_url)[:200]

In [ ]:
msg = Msg(
    name="Jarvis",
    role="assistant",
    content=[
        TextBlock(
            type="text",
            text="这是一个包含 base64 编码数据的多模态消息。\n",
        ),
        ThinkingBlock(
            type="thinking",
            thinking="我正在为 AgentScope 构建一个思考块的示例。",
        ),
        ImageBlock(
            type="image",
            source=Base64Source(
                type="base64",
                media_type="image/jpeg",
                data= image2base(img_url),
            ),
        ),
        TextBlock(
            type="text",
            text="这个是第二张图片：",
        ),
        ImageBlock(
            type="image",
            source=Base64Source(
                type="base64",
                media_type="image/jpeg",
                data= image2base(img_url),
            ),
        ),
        TextBlock(
            type="text",
            text="请描述这2个图片",
        ),
    ],
    metadata= {
        "inactivate" : False
    }
)

In [ ]:
formatter = OpenAIChatFormatter()
formatter2 = OpenAIMultiAgentFormatter()
processor = AutoProcessor.from_pretrained("/data/home/zhangchen/models/Qwen3-VL-30B-A3B-Thinking")

In [ ]:
prompt = await formatter.format([msg])

In [ ]:

for i in  prompt:
    contonts = i['content']
    contonts_new = []
    for j in contonts:
        if 'image_url' in j:
            j['image_url'] = j['image_url']['url']


In [ ]:
prompt2 = await formatter2.format([msg])

In [ ]:
prompt2

In [ ]:
prompt[0]['content'][1]['image_url'][:200]

In [ ]:
inputs = processor.apply_chat_template(
    prompt,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

In [ ]:
inputs.input_ids.shape

In [ ]:
text = processor.apply_chat_template(
    prompt,
    tokenize=False,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)
text

In [ ]:
processor.apply_chat_template(
    prompt2,
    tokenize=False,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

In [ ]:
images, videos = process_vision_info(prompt, image_patch_size=16)

In [ ]:
images

In [ ]:
inputs

In [ ]:
videos

In [ ]:
inputs_all = processor(text=text, images=images, videos=videos, do_resize=False, return_tensors="pt")

In [ ]:
inputs_all.keys()

In [ ]:
inputs_all.input_ids.shape

In [ ]:

inputs_all.attention_mask.shape

In [ ]:

inputs_all.pixel_values.shape

In [ ]:

inputs_all.image_grid_thw.shape

# Test Router

In [ ]:

router = ReActAgent(
    name="Router",
    sys_prompt="你是一个路由智能体。你的目标是将用户查询路由到正确的后续任务，注意你不需要回答用户的问题。",
    model=OpenAIChatModel(
        model_name="Qwen3-VL-30B-A3B-Thinking",
        api_key=111,
        client_kwargs={
            "base_url": "http://59.77.5.88:7999/v1"
        },
        stream=True,
    ),
    formatter=OpenAIChatFormatter(),
)


In [ ]:

# 使用结构化输出指定路由任务
class RoutingChoice(BaseModel):
    your_choice: Literal[
        "Content Generation",
        "Programming",
        "Information Retrieval",
        None,
    ] = Field(
        description="选择正确的后续任务，如果任务太简单或没有合适的任务，则选择 ``None``",
    )
    task_description: str | None = Field(
        description="任务描述",
        default=None,
    )


async def example_router_explicit() -> None:
    """使用结构化输出进行显式路由的示例。"""
    msg_user = Msg(
        "user",
        "帮我写一首诗",
        "user",
    )

    # 路由查询
    msg_res = await router(
        msg_user,
        structured_model=RoutingChoice,
    )

    # 结构化输出存储在 metadata 字段中
    # print("结构化输出：")
    # print(json.dumps(msg_res.metadata, indent=4, ensure_ascii=False))
    # print(msg_res)
    return msg_res
res = await example_router_explicit()

In [ ]:
await router.memory.get_memory()

In [ ]:
res.content

In [ ]:
res_2 = await router(msg)

In [ ]:
res_2

In [ ]:
res.content

In [ ]:
router.toolkit.get_json_schemas()

In [ ]:
history = await router.memory.get_memory()

In [ ]:
len(history)

In [ ]:
res

In [ ]:
out = await router(msg)

In [ ]:
out.content[1]

In [ ]:
h = await router.memory.get_memory()

In [ ]:
h[2]

In [ ]:
for i in h:
    print(i.content)

In [ ]:

router = ReActAgent(
    name="Router",
    sys_prompt="你是一个智能体。",
    model=OpenAIChatModel(
        model_name="Qwen3-VL-30B-A3B-Thinking",
        api_key=111,
        client_kwargs={
            "base_url": "http://59.77.5.88:7999/v1",
        },
        generate_kwargs={
            # OpenAI ChatCompletions 标准参数：能拿到输出 token 的 logprobs
            "logprobs": True,
            "top_logprobs": 5,

            # vLLM 扩展参数：通过 extra_body 传
            "extra_body": {
                "return_token_ids": True,               # 让响应里带 prompt_token_ids / token_ids :contentReference[oaicite:1]{index=1}
                # "return_tokens_as_token_ids": True,   # 可选：logprobs 里 token 变成 token_id:xxx :contentReference[oaicite:2]{index=2}
            },
        },
        stream=False,
    ),
    formatter=OpenAIChatFormatter(),
)


In [ ]:
client = openai.AsyncClient(
            api_key=1111,
            base_url='http://59.77.5.88:7999/v1'
        )

In [ ]:
openai_msg = Msg(
        "user",
        "帮我写一首诗",
        "user",
    )

In [ ]:
response = await client.chat.completions.create(
    model='Qwen3-VL-30B-A3B-Thinking',
    messages=await formatter.format([openai_msg]),
    stream=False,
    logprobs=True,
    top_logprobs=5,
    extra_body={
        "top_k": 20,
        "repetition_penalty": 1.0,
        "presence_penalty": 0.0,
        "temperature": 1.0,
        "greedy": "false",
        "seed": 1234,
        "top_p": 0.95,
        "return_token_ids": True,
        "prompt_logprobs": 5, 
        # "reasoning": {"effort": "low"}
    },
)

In [ ]:
response


In [ ]:
response.choices[0].token_ids

In [ ]:
response.choices[0].prompt_token_ids

In [ ]:
resp = await client.chat.completions.create(
    model="Qwen3-VL-30B-A3B-Thinking",
    messages=[{"role": "user", "content": "你好，讲个笑话"}],
    extra_body={
        "prompt_logprobs": 5,
        "return_token_ids": True
        },  # vLLM 扩展
)

In [ ]:
resp

In [ ]:
print(resp.choices[0].token_ids)

In [ ]:
print(resp.choices[0].message.content)

In [ ]:
print(resp.choices[0].message.reasoning)

In [ ]:
print(resp.choices[0])

In [ ]:
resp.prompt_token_ids

In [ ]:
resp.prompt_logprobs

In [ ]:
response.prompt_token_ids

In [ ]:
len(response.prompt_token_ids)

In [ ]:
response.prompt_logprobs[1]

In [ ]:
response.prompt_logprobs[3]

In [ ]:
len(response.prompt_logprobs)

# Test crop tools

In [ ]:
import math
from typing import Optional, Union, Tuple, List, Any, Dict
from concurrent.futures import ThreadPoolExecutor
from io import BytesIO
import base64
import copy

In [ ]:
MAX_RATIO = 200
SPATIAL_MERGE_SIZE = 2
IMAGE_MIN_TOKEN_NUM = 4
IMAGE_MAX_TOKEN_NUM = 16384
VIDEO_MIN_TOKEN_NUM = 128
VIDEO_MAX_TOKEN_NUM = 768

FPS = 2.0
FRAME_FACTOR = 2
FPS_MIN_FRAMES = 4
FPS_MAX_FRAMES = 768
MAX_NUM_WORKERS_FETCH_VIDEO = 8

In [ ]:
def round_by_factor(number: int, factor: int) -> int:
    """Returns the closest integer to 'number' that is divisible by 'factor'."""
    return round(number / factor) * factor


def ceil_by_factor(number: int, factor: int) -> int:
    """Returns the smallest integer greater than or equal to 'number' that is divisible by 'factor'."""
    return math.ceil(number / factor) * factor


def floor_by_factor(number: int, factor: int) -> int:
    """Returns the largest integer less than or equal to 'number' that is divisible by 'factor'."""
    return math.floor(number / factor) * factor

In [ ]:
def smart_resize(height: int, width: int, factor: int) -> Tuple[int, int]:
    """
    Rescales the image so that the following conditions are met:

    1. Both dimensions (height and width) are divisible by 'factor'.
    2. The total number of pixels is within the range ['min_pixels', 'max_pixels'].
    3. The aspect ratio of the image is maintained as closely as possible.
    """
    # max_pixels = max_pixels if max_pixels is not None else (IMAGE_MAX_TOKEN_NUM * factor ** 2)
    # min_pixels = min_pixels if min_pixels is not None else (IMAGE_MIN_TOKEN_NUM * factor ** 2)
    # assert max_pixels >= min_pixels, "The max_pixels of image must be greater than or equal to min_pixels."
    # if max(height, width) / min(height, width) > MAX_RATIO:
    #     raise ValueError(
    #         f"absolute aspect ratio must be smaller than {MAX_RATIO}, got {max(height, width) / min(height, width)}"
    #     )
    h_bar = max(factor, round_by_factor(height, factor))
    w_bar = max(factor, round_by_factor(width, factor))
    # if h_bar * w_bar > max_pixels:
    #     beta = math.sqrt((height * width) / max_pixels)
    #     h_bar = floor_by_factor(height / beta, factor)
    #     w_bar = floor_by_factor(width / beta, factor)
    # elif h_bar * w_bar < min_pixels:
    #     beta = math.sqrt(min_pixels / (height * width))
    #     h_bar = ceil_by_factor(height * beta, factor)
    #     w_bar = ceil_by_factor(width * beta, factor)
    return h_bar, w_bar

In [ ]:
def to_rgb(pil_image: Image.Image) -> Image.Image:
      if pil_image.mode == 'RGBA':
          white_background = Image.new("RGB", pil_image.size, (255, 255, 255))
          white_background.paste(pil_image, mask=pil_image.split()[3])  # Use alpha channel as mask
          return white_background
      else:
          return pil_image.convert("RGB")

In [ ]:
with open('/data/home/zhangchen/project/RL/SlideReasoner/datasets/Clv1_Vev2_Sev2_ReRev3/summary/summary.json', 'r', encoding='utf-8') as f:
    Clv1_Vev2_Sev2_ReRev3 = json.load(f)
with open('/data/home/zhangchen/project/RL/SlideReasoner/datasets/resized_image_qwen3vl/mpp_20/meta_all.json', 'r', encoding='utf-8') as f:
   image_mpp_20_meta = json.load(f)
with open('/data/home/zhangchen/project/RL/SlideReasoner/datasets/resized_image_qwen3vl/mpp_10/meta_all.json', 'r', encoding='utf-8') as f:
   image_mpp_10_meta = json.load(f)

In [ ]:
WSI_id = 'TCGA-BH-A0EB'

In [ ]:
image_mpp_20_meta['TCGA-BH-A0EB']

In [ ]:
# data_root = "/data/home/zhangchen/project/RL/SlideReasoner/datasets/Clv1_Vev2_Sev2_ReRev3_32B_MPP10"
# summary = {}
# for file_name in os.listdir(data_root):
#     if file_name == "summary":
#         continue

#     file_path = os.path.join(data_root, file_name)
#     with open(file_path, 'r', encoding='utf-8') as f:
#         file_json = json.load(f)
#     summary[file_json['case_id']] = file_json

# with open("/data/home/zhangchen/project/RL/SlideReasoner/datasets/summary/Clv1_Vev2_Sev2_ReRev3_32B_MPP10.json", 'w', encoding='utf-8') as f:
#     json.dump(summary, f, ensure_ascii=False, indent=4)

In [ ]:
BRCA_root = "/data/home/zhangchen/datasets/TCGA/WSI/BRCA_svs"
output_root = "/data/home/zhangchen/project/RL/SlideReasoner/result/test_task"

In [ ]:
os.makedirs(os.path.join(output_root, WSI_id), exist_ok=True)

In [ ]:
output_WSI_root = os.path.join(output_root, WSI_id)

In [ ]:
wsi_file_path = os.path.join(BRCA_root, image_mpp_20_meta[WSI_id]['wsi_file_name'])

In [ ]:
slide = openslide.OpenSlide(wsi_file_path)

In [ ]:
slide.dimensions

In [ ]:
slide.level_downsamples

In [ ]:
img_url = "/data/home/zhangchen/project/RL/SlideReasoner/slidereasoner/reasource/qwen3vl_assets/spatial_understanding/dining_table.png"
img = Image.open(BytesIO(base64.b64decode(image2base(img_url))))
display(img)

In [ ]:
type(img)

In [ ]:
img.size

In [ ]:
image = to_rgb(img)
display(image)
image.size

In [ ]:
def smart_resize(height: int, width: int, factor: int) -> Tuple[int, int]:
    """
    Rescales the image so that the following conditions are met:

    1. Both dimensions (height and width) are divisible by 'factor'.
    2. The total number of pixels is within the range ['min_pixels', 'max_pixels'].
    3. The aspect ratio of the image is maintained as closely as possible.
    """
    max_pixels =  (IMAGE_MAX_TOKEN_NUM * factor ** 2)
    min_pixels =  (IMAGE_MIN_TOKEN_NUM * factor ** 2)
    assert max_pixels >= min_pixels, "The max_pixels of image must be greater than or equal to min_pixels."
    if max(height, width) / min(height, width) > MAX_RATIO:
        raise ValueError(
            f"absolute aspect ratio must be smaller than {MAX_RATIO}, got {max(height, width) / min(height, width)}"
        )
    h_bar = max(factor, round_by_factor(height, factor))
    w_bar = max(factor, round_by_factor(width, factor))
    return h_bar, w_bar

In [ ]:
def validate_MinMax_pixels(bbox, w_bar, h_bar, image_idx, origin_mag, target_mag, max_pixels, min_pixels):
    
    rel_x1, rel_y1, rel_x2, rel_y2 = bbox

    pixels = w_bar * h_bar
    coord_note = (
        "Qwen3-VL bbox coords are relative on a 0-1000 scale "
        "Convert to pixels: abs_px = rel/1000 * (img_w,img_h)."
    )    
    if pixels > max_pixels:
        raise ValueError(
            "WSI_PATCH_TOO_LARGE: "
            f"image_idx={image_idx}; "
            f"origin_wsi_magnification=x{origin_mag}; "
            f"target_wsi_magnification=x{target_mag}; "
            f"bbox_2d=[{rel_x1},{rel_y1},{rel_x2},{rel_y2}]; "
            f"corresponding_image_wh={w_bar}x{h_bar}; "
            f"patch_pixels={pixels} > max_pixels={max_pixels}; "
            f"coord_note={coord_note} "
            "Action: shrink bbox (reduce area) and retry with same image_idx & target_wsi_magnification."
        )

    elif pixels < min_pixels:
        raise ValueError(
            "WSI_PATCH_TOO_SMALL: "
            f"image_idx={image_idx}; "
            f"origin_wsi_magnification=x{origin_mag}; "
            f"target_wsi_magnification=x{target_mag}; "
            f"bbox_2d=[{rel_x1},{rel_y1},{rel_x2},{rel_y2}]; "
            f"corresponding_image_wh={w_bar}x{h_bar}; "
            f"patch_pixels={pixels} < min_pixels={min_pixels}; "
            f"coord_note={coord_note} "
            "Action: expand_bbox (increase area) and retry with same image_idx & target_wsi_magnification."
        )

In [ ]:
def validate_MinMax_pixels_test(bbox, w_bar, h_bar, max_pixels, min_pixels):
    
    rel_x1, rel_y1, rel_x2, rel_y2 = bbox

    pixels = w_bar * h_bar
    coord_note = (
        "Qwen3-VL bbox coords are relative on a 0-1000 scale "
        "Convert to pixels: abs_px = rel/1000 * (img_w,img_h)."
    )    
    if pixels > max_pixels:
        raise ValueError(
            "BBOX_PATCH_TOO_LARGE: "
            f"bbox_2d=[{rel_x1},{rel_y1},{rel_x2},{rel_y2}]; "
            f"corresponding_image_wh={w_bar}x{h_bar}; "
            f"patch_pixels={pixels} > max_pixels={max_pixels}; "
            f"coord_note={coord_note} "
            "Action: shrink bbox (reduce area)."
        )

    elif pixels < min_pixels:
        raise ValueError(
            "BBOX_PATCH_TOO_SMALL: "
            f"bbox_2d=[{rel_x1},{rel_y1},{rel_x2},{rel_y2}]; "
            f"corresponding_image_wh={w_bar}x{h_bar}; "
            f"patch_pixels={pixels} < min_pixels={min_pixels}; "
            f"coord_note={coord_note} "
            "Action: expand_bbox (increase area)."
        )

In [ ]:
width, height = image.size
print(width, height)

In [ ]:
resized_height, resized_width = smart_resize(height, width, factor=32)

In [ ]:
print(resized_height, resized_width)

In [ ]:
def maybe_resize_bbox(left, top, right, bottom, img_width, img_height):
    """Resize bbox to ensure it's valid"""
    left = max(0, left)
    top = max(0, top)
    right = min(img_width, right)
    bottom = min(img_height, bottom)

    height = bottom - top
    width = right - left
    if height < 32 or width < 32:
        center_x = (left + right) / 2.0
        center_y = (top + bottom) / 2.0
        ratio = 32 / min(height, width)
        new_half_height = ceil(height * ratio * 0.5)
        new_half_width = ceil(width * ratio * 0.5)
        new_left = floor(center_x - new_half_width)
        new_right = ceil(center_x + new_half_width)
        new_top = floor(center_y - new_half_height)
        new_bottom = ceil(center_y + new_half_height)

        # Ensure the resized bbox is within image bounds
        new_left = max(0, new_left)
        new_top = max(0, new_top)
        new_right = min(img_width, new_right)
        new_bottom = min(img_height, new_bottom)

        new_height = new_bottom - new_top
        new_width = new_right - new_left

        if new_height >= 32 and new_width >= 32:
            return [new_left, new_top, new_right, new_bottom]
        else:
            raise ValueError(f"new_height: {new_height} or new_width: {new_width} < 32, need to increase the bbox area ")
    return [left, top, right, bottom]

In [ ]:
obersvaton_list = []
work_dir = "/data/home/zhangchen/project/RL/SlideReasoner/result/test_task/TCGA-BH-A0EB"
action_idx = 1
def zoom_in_image(
    bbox_2d: Annotated[list[float], Field(min_length=4, max_length=4)],
    label: str,
    observation_index: Annotated[int, Field(ge=0, description="Source image index starting from 0.")],
) -> ToolResponse:

    """Zoom in on a specific region of an image by cropping it based on a bounding box (bbox) and an optional object label.

    Args:
        bbox_2d (`list[float]`):
            The bounding box of the region to zoom in, as [x1, y1, x2, y2], where (x1, y1) is the left-top corner (x1 is left and y1 is top) and (x2, y2) is the right-bottom corner (x2 is right and y2 is bottom). The bounding box uses the relative coordinated with range 0-1000.
        label (`str`):
            The name or label of the object in the specified bounding box.
        observation_index (`int`):
            The index of the image to zoom-in(starting from 0). The index of the image to crop. Choose 1 to operate on original image.

    Returns:
        `ToolResponse`:
            The tool response containing the result of the writing operation.
    """
    try: 
        img_file_path = obersvaton_list[observation_index]

        if not os.path.exists(img_file_path):
            raise ValueError(f'img_file_path: {img_file_path} is not exist')

        original_image = to_rgb(Image.open(img_file_path))

        img_width, img_height  = original_image.size

        rel_x1, rel_y1, rel_x2, rel_y2 = bbox_2d

        abs_x1 = floor(rel_x1 / 1000. * img_width)
        abs_y1 = floor(rel_y1 / 1000. * img_height)
        abs_x2 = ceil(rel_x2 / 1000. * img_width)
        abs_y2 = ceil(rel_y2 / 1000. * img_height)

        validated_bbox = maybe_resize_bbox(abs_x1, abs_y1, abs_x2, abs_y2, img_width, img_height)


        left, top, right, bottom = validated_bbox


        # Crop the image

        cropped_image = original_image.crop((left, top, right, bottom))

        new_w, new_h = smart_resize((right - left), (bottom - top), factor=32)

        validate_MinMax_pixels_test(bbox_2d, new_w, new_h, max_pixels= 16384 * 32 * 32, min_pixels= 4 * 32 * 32)    
        
        cropped_image = cropped_image.resize((new_w, new_h), resample=Image.BICUBIC)

        # save crop image

        output_path = os.path.abspath(os.path.join(work_dir, f'action_{action_idx}_{uuid.uuid4()}.png'))
        cropped_image.save(output_path)

        new_img_idx = len(obersvaton_list)
        obersvaton_list.append(output_path)
        return ToolResponse(
        content=[
            TextBlock(
                type="text",
                text=(
                    "zoom_in_image succeeded.\n"
                    "Generated a zoomed-in ROI view (image) from the selected bounding box.\n"
                    f"- returned observation_index: {new_img_idx}\n"
                    f"- source observation_index: {observation_index}\n"
                    f"- label: {label}\n"
                    f"To reference this ROI view in later tool calls, use observation_index={new_img_idx}."
                ),
            ),
            ImageBlock(
                type="image",
                source=URLSource(
                    type="url",
                    url=output_path
                )
            )
        ],
        metadata={
            "success": True,
            # "observation_index": [new_img_idx],
            # "source_observation_index" : [observation_index]
            }
        )
    
    except Exception as e:
        obs = f'Tool Execution Error {str(e)}'
        return ToolResponse(
        content=[

            TextBlock(
                type="text",
                text=f"Failure to execute zoom_in_image, error: {obs}.",
            )
        ],
        metadata={"success": False}
        )



In [ ]:
tool_response=zoom_in_image(bbox_2d=[290, 400, 300, 600], label='test', img_idx=0)

In [ ]:
tool_response

In [ ]:
tool_response.content 

In [ ]:
img_url

In [ ]:
toolkit = Toolkit()

In [ ]:
toolkit.register_tool_function(zoom_in_image)
# toolkit.register_tool_function(write_text_file)

In [ ]:
toolkit.remove_tool_function('zoom_in_image')

In [ ]:
toolkit.get_json_schemas()

In [ ]:
toolkit.tools

# Test Multimodel_print

In [ ]:
_stream_prefix = {}

def _print_text_block(
    _stream_prefix,
    msg_id: str,
    name_prefix: str,
    text_content: str,
    thinking_and_text_to_print: list[str],
) -> None:
    """Print the text block and thinking block content.

    Args:
        msg_id (`str`):
            The unique identifier of the message
        name_prefix (`str`):
            The prefix for the message, e.g. "{name}: " for text block and
            "{name}(thinking): " for thinking block.
        text_content (`str`):
            The textual content to be printed.
        thinking_and_text_to_print (`list[str]`):
            A list of textual content to be printed together. Here we
            gather the text and thinking blocks to print them together.
    """
    thinking_and_text_to_print.append(
        f"{name_prefix}: {text_content}",
    )
    # The accumulated text and thinking blocks to print
    to_print = "\n".join(thinking_and_text_to_print)

    # The text prefix that has been printed
    if msg_id not in _stream_prefix:
        _stream_prefix[msg_id] = {}

    text_prefix = _stream_prefix[msg_id].get("text", "")

    # Only print when there is new text content
    if len(to_print) > len(text_prefix):
        print(to_print[len(text_prefix) :], end="")

        # Save the printed text prefix
        _stream_prefix[msg_id]["text"] = to_print

def _in_jupyter() -> bool:
    try:
        from IPython import get_ipython  # type: ignore
        ip = get_ipython()
        return ip is not None and "IPKernelApp" in ip.config
    except Exception:
        return False

def _resize_keep_ratio_max_side(img, max_side: int = 400):
    """Keep aspect ratio; ensure longest side <= max_side."""
    if not max_side or max_side <= 0:
        return img

    try:
        resample = __import__("PIL.Image").Image.Resampling.LANCZOS  # Pillow >= 9
    except Exception:
        from PIL import Image  # type: ignore
        resample = getattr(Image, "LANCZOS", 1)

    w, h = img.size
    if max(w, h) <= max_side:
        return img

    img = img.copy()
    img.thumbnail((max_side, max_side), resample=resample)
    return img

def _display_image_from_source(
    source: Dict[str, Any],
    print_image: bool = True,
    save_dir: Optional[str] = None,
) -> Optional[str]:
    """
    Returns:
        - Jupyter: None (already displayed)
        - Terminal: returns file path if saved, or None if only printed URL
    """
    if not print_image:
        return None

    src_type = source.get("type")
    if src_type not in ("url", "base64"):
        raise ValueError(f"image source type must be 'url' or 'base64', got: {src_type}")

    is_jup = _in_jupyter()

    # Lazy imports (optional deps)
    PIL_OK = True
    try:
        from PIL import Image  # type: ignore
    except Exception:
        PIL_OK = False

    if src_type == "url":
        url = source.get("url")
        if not url:
            return None

        if is_jup and PIL_OK:
            try:
                import requests  # type: ignore
                from IPython.display import display  # type: ignore

                # img = Image.open(requests.get(url, stream=True, timeout=20).raw)
                img = Image.open(url)
                img = _resize_keep_ratio_max_side(img)
                display(img)
                return None
            except Exception as e:
                print(f"[IMAGE] failed to display from url: {url} ({e})")
                print(f"[IMAGE] {url}")
                return None

        # terminal / no PIL
        print(f"[IMAGE] {url}")
        return None
    else:
        raise NotImplementedError(f"src_type : {src_type} is not implemented")
    
def _print_last_block(
    _stream_prefix,
    block: ToolUseBlock
    | ToolResultBlock
    | ImageBlock
    | VideoBlock
    | AudioBlock,
    msg: Msg,
    print_image: bool = False
) -> None:
    """Process and print the last content block, and the block type
    is not text, or thinking.

    Args:
        block (`ToolUseBlock | ToolResultBlock | ImageBlock | VideoBlock \
        | AudioBlock`):
            The content block to be printed
        msg (`Msg`):
            The message object
    """
    # TODO: We should consider how to handle the multimodal blocks in the
    #  terminal, since the base64 data may be too long to display.

    if block.get("type") == "image":
        if print_image:
            block_source =  block.get("source")

            _display_image_from_source(block_source, print_image=print_image)
        else:
            return
    
    if block.get("type") in ["video", "audio"]:
            return

    text_prefix = _stream_prefix.get(msg.id, {}).get("text", "")

    if text_prefix:
        # Add a newline to separate from previous text content
        print_newline = "" if text_prefix.endswith("\n") else "\n"
        print(
            f"{print_newline}"
            f"{json.dumps(block, indent=4, ensure_ascii=False)}",
        )
    else:
        print(
            f"{msg.name}:"
            f" {json.dumps(block, indent=4, ensure_ascii=False)}",
        )


async def print_test(
        msg: Msg,
        _stream_prefix: Dict,
        last: bool = True,
        print_image=False
) -> None:
    """The function to display the message.

    Args:
        msg (`Msg`):
            The message object to be printed.
        last (`bool`, defaults to `True`):
            Whether this is the last one in streaming messages. For
            non-streaming message, this should always be `True`.
    """

    # The accumulated textual content to print, including the text blocks and the thinking blocks
    thinking_and_text_to_print = []
    # Todo: We need to handle the multimodal blocks in terminal (images like qwen-agent)

    for block in msg.get_content_blocks():
        if block["type"] == "text":
            _print_text_block(
                _stream_prefix,
                msg.id,
                name_prefix=msg.name,
                text_content=block["text"],
                thinking_and_text_to_print=thinking_and_text_to_print,
            )
    

        elif block["type"] == "thinking":
            _print_text_block(
                _stream_prefix,
                msg.id,
                name_prefix=f"{msg.name}(thinking)",
                text_content=block["thinking"],
                thinking_and_text_to_print=thinking_and_text_to_print,
            )
        
        elif last:
            _print_last_block(_stream_prefix, block, msg, print_image=print_image)


    # Clean up resources if this is the last message in streaming
    if last and msg.id in _stream_prefix:
        stream_prefix = _stream_prefix.pop(msg.id)
        if "text" in stream_prefix and not stream_prefix["text"].endswith(
            "\n",
        ):
            print()

# Test SimpleReAct

In [ ]:
from agentscope.agent import ReActAgent, AgentBase, ReActAgentBase
from agentscope.formatter import OpenAIChatFormatter, FormatterBase
from agentscope.memory import InMemoryMemory, MemoryBase
from agentscope.message import Msg, AudioBlock, ToolResultBlock, ToolUseBlock, ImageBlock, VideoBlock, URLSource
from agentscope.tool import Toolkit, ToolResponse, execute_python_code, execute_shell_command
from agentscope.model import OpenAIChatModel, ChatModelBase
from agentscope.tracing import trace_reply
from agentscope._logging import logger
import asyncio
import os
from typing import Type, Any, AsyncGenerator, Literal, Dict, Optional
from pydantic import BaseModel, ValidationError, Field
import json

## Test Step by Step

In [ ]:
analysis_prompt = """Your role is that of a research assistant specializing in visual information. Answer questions about images by looking at them closely and then using research tools. Please follow this structured thinking process and show your work.

Start an iterative loop for each question:

- **First, look closely:** Begin with a detailed description of the image, paying attention to the user's question. List what you can tell just by looking, and what you'll need to look up.
- **Next, find information:** Use a tool to research the things you need to find out.
- **Then, review the findings:** Carefully analyze what the tool tells you and decide on your next action.

Continue this loop until your research is complete.

To finish, bring everything together in a clear, synthesized answer that fully responds to the user's question."""

In [ ]:
model = OpenAIChatModel(
        model_name="Qwen3-VL-30B-A3B-Thinking",
        api_key=111,
        client_kwargs={
            "base_url": "http://59.77.5.88:7999/v1"
        },
        stream=True,
        generate_kwargs={
            # OpenAI ChatCompletions 标准参数：能拿到输出 token 的 logprobs
            "logprobs": True,
            "top_logprobs": 5,

            # vLLM 扩展参数：通过 extra_body 传
            "extra_body": {
                "return_token_ids": True,               # 让响应里带 prompt_token_ids / token_ids :contentReference[oaicite:1]{index=1}
                # "return_tokens_as_token_ids": True,   # 可选：logprobs 里 token 变成 token_id:xxx :contentReference[oaicite:2]{index=2}
            },
        },
    )
formatter = OpenAIChatFormatter()

In [ ]:
msg_user = Msg(
    "user",
    "帮我写一首诗, 并且在python上打印出来",
    "user",
)

In [ ]:
prompt = await formatter.format([msg_user])

In [ ]:
toolkit = Toolkit()
toolkit.register_tool_function(execute_python_code)

In [ ]:
toolkit.get_json_schemas()

In [ ]:
res = await model(
    prompt,
    tools=toolkit.get_json_schemas(),
    
)

In [ ]:

_stream_prefix = {}
msg = Msg(name='testagent', content=[], role="assistant")
async for content_chunk in res:
    msg.content = content_chunk.content
    await print_test(msg, _stream_prefix, False)
await print_test(msg, _stream_prefix, True)


In [ ]:
await print_test(msg, _stream_prefix, True, True)


In [ ]:
msg.content.append(image_msg)

In [ ]:
await print_test(msg, _stream_prefix, True, True)

In [ ]:
msg.content.append(
    {
        'type': 'text',
        'text' : '帮我写一首诗, 并且在python上打印出来'
    }
)

In [ ]:
img_url = '/data/home/zhangchen/project/RL/SlideReasoner/slidereasoner/reasource/qwen3vl_assets/spatial_understanding/dining_table.png'

In [ ]:
image_msg = ImageBlock(
    type="image",
    source=URLSource(
        type="url",
        url=img_url
    )
)

In [ ]:
image_msg

## Test Crop tools

In [ ]:
from qwen_agent.agents import Assistant
from qwen_agent.utils.output_beautify import typewriter_print, multimodal_typewriter_print

In [ ]:
llm_cfg = {
    # Use dashscope API
    # 'model': 'qwen3-vl-plus',
    # 'model_server': 'qwenvl_dashscope',
    # 'api_key': '' # **fill your api key here**

    # Use a model service compatible with the OpenAI API, such as vLLM or Ollama:
    'model_type': 'qwenvl_oai',
    'model': 'Qwen3-VL-30B-A3B-Thinking',
    'model_server': 'http://59.77.5.88:7999/v1',  # base_url, also known as api_base
    'api_key': 'EMPTY',
    # 'generate_cfg': {
    #     "top_p": 0.8,
    #     "top_k": 20,
    #     "temperature": 0.7,
    #     "repetition_penalty": 1.0,
    #     "presence_penalty": 1.5
    # }
}

analysis_prompt = """Your role is that of a research assistant specializing in visual information. Answer questions about images by looking at them closely and then using research tools. Please follow this structured thinking process and show your work.

Start an iterative loop for each question:

- **First, look closely:** Begin with a detailed description of the image, paying attention to the user's question. List what you can tell just by looking, and what you'll need to look up.
- **Next, find information:** Use a tool to research the things you need to find out.
- **Then, review the findings:** Carefully analyze what the tool tells you and decide on your next action.

Continue this loop until your research is complete.

To finish, bring everything together in a clear, synthesized answer that fully responds to the user's question."""

tools = ['image_zoom_in_tool']
agent = Assistant(
    llm=llm_cfg,
    function_list=tools,
    system_message=analysis_prompt,
    # [!Optional] We provide `analysis_prompt` to enable VL conduct deep analysis. Otherwise use system_message='' to simply enable the tools.
)

In [ ]:
agent.function_map

In [ ]:
agent.mem.system_files

In [ ]:
messages = []
messages += [
    {"role": "user", "content": [
        {"image": "/data/home/zhangchen/project/RL/SlideReasoner/slidereasoner/reasource/qwen3vl_assets/qwenagent/hopinn.jpg"},
        # {"text": "Where was the picture taken? "}
        # {"text": "To answer the question , you must use the image_zoom_in_tool multiple times (over 3) to closely examine specific details that can serve as clear evidence for determining the location. Where was the picture taken?"}
        {"text":"能通过多次的image_zoom_in_tool来找到这个图片拍摄在哪里的证据吗？如果截取的图片有明显的截断最好请扩大截取范围再次判断，避免重要信息的缺失， 每个重要视觉证据需要image_zoom_in_tool到对应区域二次的验证信息无误"}
    ]}
]

In [ ]:
messages

In [ ]:
response_plain_text = ''
for ret_messages in agent.run(messages):
    # `ret_messages` will contain all subsequent messages, consisting of interleaved assistant messages and tool responses
    response_plain_text = multimodal_typewriter_print(ret_messages, response_plain_text)

In [ ]:
ret_messages

In [ ]:
obersvaton_list = [
    "/data/home/zhangchen/project/RL/SlideReasoner/slidereasoner/reasource/qwen3vl_assets/qwenagent/hopinn.jpg"
]
# obersvaton_list = ["/data/home/zhangchen/project/RL/SlideReasoner/slidereasoner/reasource/qwen3vl_assets/spatial_understanding/dining_table.png"]
work_dir = "/data/home/zhangchen/project/RL/SlideReasoner/result/test_task/TCGA-BH-A0EB"
action_idx = 1
def zoom_in_image(
    bbox_2d: Annotated[list[float], Field(min_length=4, max_length=4)],
    label: str,
    observation_index: Annotated[int, Field(ge=0, description="Source image index starting from 0.")],
) -> ToolResponse:

    """Zoom in on a specific region of an image by cropping it based on a bounding box (bbox) and an optional object label

    Args:
        bbox_2d (`list[float]`):
            The bounding box of the region to zoom in, as [x1, y1, x2, y2], where (x1, y1) is the top-left corner and (x2, y2) is the bottom-right corner. The bounding box uses the relative coordinated with range 0-1000.
        label (`str`):
            The name or label of the object in the specified bounding box
        observation_index (`int`):
            The index of the image to zoom-in(starting from 0)

    Returns:
        `ToolResponse`:
            The tool response containing the result of the writing operation.
    """

    # The bounding box of the region to zoom in, as [x1, y1, x2, y2], where (x1, y1) is the left-top corner (x1 is left and y1 is top) and (x2, y2) is the right-bottom corner (x2 is right and y2 is bottom). The bounding box uses the relative coordinated with range 0-1000.
    try: 
        img_file_path = obersvaton_list[observation_index]

        if not os.path.exists(img_file_path):
            raise ValueError(f'img_file_path: {img_file_path} is not exist')

        original_image = to_rgb(Image.open(img_file_path))

        img_width, img_height  = original_image.size
        print(img_width, img_height)

        rel_x1, rel_y1, rel_x2, rel_y2 = bbox_2d

        abs_x1 = floor(rel_x1 / 1000. * img_width)
        abs_y1 = floor(rel_y1 / 1000. * img_height)
        abs_x2 = ceil(rel_x2 / 1000. * img_width)
        abs_y2 = ceil(rel_y2 / 1000. * img_height)

        validated_bbox = maybe_resize_bbox(abs_x1, abs_y1, abs_x2, abs_y2, img_width, img_height)


        left, top, right, bottom = validated_bbox


        # Crop the image

        cropped_image = original_image.crop((left, top, right, bottom))

        new_w, new_h = smart_resize((right - left), (bottom - top), factor=32)

        validate_MinMax_pixels_test(bbox_2d, new_w, new_h, max_pixels= 16384 * 32 * 32, min_pixels= 4 * 32 * 32)    
        
        cropped_image = cropped_image.resize((new_w, new_h), resample=Image.BICUBIC)

        # save crop image

        output_path = os.path.abspath(os.path.join(work_dir, f'action_{action_idx}_{uuid.uuid4()}.png'))
        cropped_image.save(output_path)

        new_img_idx = len(obersvaton_list)
        obersvaton_list.append(output_path)
        return ToolResponse(
        content=[
            TextBlock(
                type="text",
                text=(
                    "zoom_in_image succeeded.\n"
                    "Generated a zoomed-in ROI view (image) from the selected bounding box.\n"
                    f"- returned observation_index: {new_img_idx}\n"
                    f"- source observation_index: {observation_index}\n"
                    f"- label: {label}\n"
                    f"To reference this ROI view in later tool calls, use observation_index={new_img_idx}."
                ),
            ),
            ImageBlock(
                type="image",
                source=URLSource(
                    type="url",
                    url=output_path
                )
            )
        ],
        metadata={
            "success": True,
            "observation_index": [new_img_idx],
            "source_observation_index" : [observation_index],
            "label" : [label]
            }
        )
    
    except Exception as e:
        obs = f'Tool Execution Error {str(e)}'
        return ToolResponse(
        content=[

            TextBlock(
                type="text",
                text=f"Failure to execute zoom_in_image, error: {obs}.",
            )
        ],
        metadata={"success": False}
        )



In [ ]:
    # 'generate_cfg': {
    #     "top_p": 0.8,
    #     "top_k": 20,
    #     "temperature": 0.7,
    #     "repetition_penalty": 1.0,
    #     "presence_penalty": 1.5
    # }

In [ ]:
model = OpenAIChatModel(
        model_name="Qwen3-VL-30B-A3B-Thinking",
        api_key=111,
        client_kwargs={
            "base_url": "http://59.77.5.88:7999/v1"
        },
        stream=True,
        generate_kwargs={
            # OpenAI ChatCompletions 标准参数：能拿到输出 token 的 logprobs
            # "logprobs": True,
            # "top_logprobs": 5,
            # vLLM 扩展参数：通过 extra_body 传
            "extra_body": {
                "return_token_ids": True,               # 让响应里带 prompt_token_ids / token_ids :contentReference[oaicite:1]{index=1}
                # "return_tokens_as_token_ids": True,   # 可选：logprobs 里 token 变成 token_id:xxx :contentReference[oaicite:2]{index=2}

            # "top_p": 0.8,
            # "top_k": 20,
            # "temperature": 0.7,
            # "repetition_penalty": 1.0,
            # "presence_penalty": 1.5,
            },
        },
    )
formatter = OpenAIChatFormatter()

In [ ]:
toolkit = Toolkit()
toolkit.register_tool_function(zoom_in_image)
toolkit.get_json_schemas()

In [ ]:
messages_test_crop = [
    Msg(
        name="testAgent",
        content=[
            TextBlock(
                type="text",
                text=analysis_prompt
            )
        ],
        role='system'
    ),
    Msg(
        name="user",
        content=[
            TextBlock(
                type="text",
                text="Where was the picture taken?"
                # text="请crop出完整的车辆?"
            ),
            ImageBlock(
                type="image",
                source=URLSource(
                    type="url",
                    url="/data/home/zhangchen/project/RL/SlideReasoner/slidereasoner/reasource/qwen3vl_assets/qwenagent/hopinn.jpg"
                )
            )
        ],
        role="user",
    )
]

In [ ]:
messages_grouding = [
    Msg(
        name="user",
        content=[
            TextBlock(
                type="text",
                text='locate every instance that belongs to the following categories: "plate/dish, scallop, wine bottle, tv, bowl, spoon, air conditioner, coconut drink, cup, chopsticks, person". Report bbox coordinates in JSON format.'

            ),
            ImageBlock(
                type="image",
                source=URLSource(
                    type="url",
                    url="/data/home/zhangchen/project/RL/SlideReasoner/slidereasoner/reasource/qwen3vl_assets/spatial_understanding/dining_table.png"
                )
            )
        ],
        role="user",
    )
]

In [ ]:
messages_test_crop

In [ ]:
formatter = OpenAIChatFormatter()

In [ ]:
prompt_test_crop = await formatter.format(messages_test_crop)
# prompt_test_crop

In [ ]:
prompt_test_gounding = await formatter.format(messages_grouding)
# prompt_test_gounding

In [ ]:
res = await model(
    prompt_test_crop,
    tools=toolkit.get_json_schemas(),
)

In [ ]:
res = await model(
    prompt_test_gounding,
)

In [ ]:
toolkit.get_json_schemas()

In [ ]:
# @title Plotting Util

# # Get Noto JP font to display janapese characters
# !apt-get install fonts-noto-cjk  # For Noto Sans CJK JP

#!apt-get install fonts-source-han-sans-jp # For Source Han Sans (Japanese)

import json
import random
import io
import ast
from io import BytesIO
from PIL import Image, ImageDraw, ImageFont
from PIL import ImageColor
import xml.etree.ElementTree as ET

additional_colors = [colorname for (colorname, colorcode) in ImageColor.colormap.items()]

def decode_json_points(text: str):
    """Parse coordinate points from text format"""
    try:
        # 清理markdown标记
        if "```json" in text:
            text = text.split("```json")[1].split("```")[0]
        
        # 解析JSON
        data = json.loads(text)
        points = []
        labels = []
        
        for item in data:
            if "point_2d" in item:
                x, y = item["point_2d"]
                points.append([x, y])
                
                # 获取label，如果没有则使用默认值
                label = item.get("label", f"point_{len(points)}")
                labels.append(label)
        
        return points, labels
        
    except Exception as e:
        print(f"Error: {e}")
        return [], []
        

def plot_bounding_boxes(im, bounding_boxes):
    """
    Plots bounding boxes on an image with markers for each a name, using PIL, normalized coordinates, and different colors.

    Args:
        img_path: The path to the image file.
        bounding_boxes: A list of bounding boxes containing the name of the object
         and their positions in normalized [y1 x1 y2 x2] format.
    """

    # Load the image
    img = im
    width, height = img.size
    print(img.size)
    # Create a drawing object
    draw = ImageDraw.Draw(img)

    # Define a list of colors
    colors = [
    'red',
    'green',
    'blue',
    'yellow',
    'orange',
    'pink',
    'purple',
    'brown',
    'gray',
    'beige',
    'turquoise',
    'cyan',
    'magenta',
    'lime',
    'navy',
    'maroon',
    'teal',
    'olive',
    'coral',
    'lavender',
    'violet',
    'gold',
    'silver',
    ] + additional_colors

    # Parsing out the markdown fencing
    bounding_boxes = parse_json(bounding_boxes)

    # font = ImageFont.truetype("NotoSansCJK-Regular.ttc", size=14)

    try:
      json_output = ast.literal_eval(bounding_boxes)
    except Exception as e:
      end_idx = bounding_boxes.rfind('"}') + len('"}')
      truncated_text = bounding_boxes[:end_idx] + "]"
      json_output = ast.literal_eval(truncated_text)

    if not isinstance(json_output, list):
      json_output = [json_output]

    # Iterate over the bounding boxes
    for i, bounding_box in enumerate(json_output):
      # Select a color from the list
      color = colors[i % len(colors)]

      # Convert normalized coordinates to absolute coordinates
      abs_y1 = int(bounding_box["bbox_2d"][1] / 1000 * height)
      abs_x1 = int(bounding_box["bbox_2d"][0] / 1000 * width)
      abs_y2 = int(bounding_box["bbox_2d"][3] / 1000 * height)
      abs_x2 = int(bounding_box["bbox_2d"][2] / 1000 * width)

      if abs_x1 > abs_x2:
        abs_x1, abs_x2 = abs_x2, abs_x1

      if abs_y1 > abs_y2:
        abs_y1, abs_y2 = abs_y2, abs_y1

      # Draw the bounding box
      draw.rectangle(
          ((abs_x1, abs_y1), (abs_x2, abs_y2)), outline=color, width=3
      )

      # Draw the text
      if "label" in bounding_box:
        draw.text((abs_x1 + 8, abs_y1 + 6), bounding_box["label"], fill=color)

    # Display the image
    img.show()


def plot_points(im, text):
  img = im
  width, height = img.size
  draw = ImageDraw.Draw(img)
  colors = [
    'red', 'green', 'blue', 'yellow', 'orange', 'pink', 'purple', 'brown', 'gray',
    'beige', 'turquoise', 'cyan', 'magenta', 'lime', 'navy', 'maroon', 'teal',
    'olive', 'coral', 'lavender', 'violet', 'gold', 'silver',
  ] + additional_colors

  points, descriptions = decode_json_points(text)
  print("Parsed points: ", points)
  print("Parsed descriptions: ", descriptions)
  if points is None or len(points) == 0:
    img.show()
    return

#   font = ImageFont.truetype("NotoSansCJK-Regular.ttc", size=14)

  for i, point in enumerate(points):
    color = colors[i % len(colors)]
    abs_x1 = int(point[0])/1000 * width
    abs_y1 = int(point[1])/1000 * height
    radius = 2
    draw.ellipse([(abs_x1 - radius, abs_y1 - radius), (abs_x1 + radius, abs_y1 + radius)], fill=color)
    draw.text((abs_x1 - 20, abs_y1 + 6), descriptions[i], fill=color)
  
  img.show()

def plot_points_json(im, text):
  img = im
  width, height = img.size
  draw = ImageDraw.Draw(img)
  colors = [
    'red', 'green', 'blue', 'yellow', 'orange', 'pink', 'purple', 'brown', 'gray',
    'beige', 'turquoise', 'cyan', 'magenta', 'lime', 'navy', 'maroon', 'teal',
    'olive', 'coral', 'lavender', 'violet', 'gold', 'silver',
  ] + additional_colors
#   font = ImageFont.truetype("NotoSansCJK-Regular.ttc", size=14)

  text = text.replace('```json', '')
  text = text.replace('```', '')
  data = json.loads(text)
  for item in data:
    point_2d = item['point_2d']
    label = item['label']
    x, y = int(point_2d[0] / 1000 * width), int(point_2d[1] / 1000 * height)
    radius = 2
    draw.ellipse([(x - radius, y - radius), (x + radius, y + radius)], fill=colors[0])
    draw.text((x + 2*radius, y + 2*radius), label, fill=colors[0])
  
  img.show()
  
  
  

# @title Parsing JSON output
def parse_json(json_output):
    # Parsing out the markdown fencing
    lines = json_output.splitlines()
    for i, line in enumerate(lines):
        if line == "```json":
            json_output = "\n".join(lines[i+1:])  # Remove everything before "```json"
            json_output = json_output.split("```")[0]  # Remove everything after the closing "```"
            break  # Exit the loop once "```json" is found
    return json_output

In [ ]:
import requests

import os
import copy
import traceback
import time
from openai import OpenAI



def inference_with_openai_api(img_url, prompt, min_pixels=64 * 32 * 32, max_pixels=9800* 32 * 32):
    import base64
    import os
    if os.path.exists(img_url):
        with open(img_url, "rb") as image_file:
            base64_image = base64.b64encode(image_file.read()).decode("utf-8")
    elif img_url.startswith("http://") or img_url.startswith("https://"):
        response = requests.get(img_url)
        response.raise_for_status()
        base64_image = base64.b64encode(response.content).decode("utf-8")
    else:
        raise ValueError("Invalid image URL")
    client = OpenAI(
        api_key="Empty",
        base_url="http://59.77.5.88:7999/v1"
    )
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}"
                    },
                    "min_pixels": min_pixels,
                    "max_pixels": max_pixels
                },
                {"type": "text", "text": prompt},
            ],
        }
    ]
    completion = client.chat.completions.create(
        model="Qwen3-VL-30B-A3B-Thinking",  # 可按需更换模型名称。模型列表：https://help.aliyun.com/zh/model-studio/models
        messages=messages,
    )
    return completion.choices[0].message.content

In [ ]:
msg.content[0]['text']

In [ ]:
_stream_prefix = {}
msg = Msg(name='testagent', content=[], role="assistant")
async for content_chunk in res:
    msg.content = content_chunk.content
    await print_test(msg, _stream_prefix, False)
await print_test(msg, _stream_prefix, True)


In [ ]:
tool_use_blocks: list = msg.get_content_blocks(
                    "tool_use",
                )

In [ ]:
for tool_call in tool_use_blocks:
    print(tool_call)
    tool_res = await toolkit.call_tool_function(tool_call)
    print(tool_res)

In [ ]:
tool_res_msg = Msg(
    "system",
    [
        ToolResultBlock(
            type="tool_result",
            id=tool_call["id"],
            name=tool_call["name"],
            output=[],
        ),
    ],
    "system",
)
async for chunk in tool_res:
    # Turn into a tool result block
    tool_res_msg.content[0][  # type: ignore[index]
        "output"
    ] = chunk.content

In [ ]:
msg

In [ ]:
msg.content

In [ ]:
image = Image.open("/data/home/zhangchen/project/RL/SlideReasoner/slidereasoner/reasource/qwen3vl_assets/spatial_understanding/dining_table.png")

image.thumbnail([640,640], Image.Resampling.LANCZOS)
plot_bounding_boxes(image, msg.content[-1]['text'])

In [ ]:
prompt = 'locate every instance that belongs to the following categories: "plate/dish, scallop, wine bottle, tv, bowl, spoon, air conditioner, coconut drink, cup, chopsticks, person". Report bbox coordinates in JSON format.'
img_url = "/data/home/zhangchen/project/RL/SlideReasoner/slidereasoner/reasource/qwen3vl_assets/spatial_understanding/dining_table.png"
model_response = inference_with_openai_api(img_url, prompt)
print(model_response)
image = Image.open(img_url)

image.thumbnail([640,640], Image.Resampling.LANCZOS)
plot_bounding_boxes(image, model_response)

In [ ]:
image.thumbnail([640,640], Image.Resampling.LANCZOS)
plot_bounding_boxes(image, model_response)

In [ ]:
tool_use_blocks: list = msg.get_content_blocks(
                    "tool_use",
                )

In [ ]:
tool_res

In [ ]:
# tool_call['input']['bbox_2d'] = [584, 496, 651, 660]

In [ ]:
tool_res_msg = Msg(
    "system",
    [
        ToolResultBlock(
            type="tool_result",
            id=tool_call["id"],
            name=tool_call["name"],
            output=[],
        ),
    ],
    "system",
)
async for chunk in tool_res:
    # Turn into a tool result block
    tool_res_msg.content[0][  # type: ignore[index]
        "output"
    ] = chunk.content

In [ ]:
tool_res_msg

In [ ]:
class SimpleReActAgent(ReActAgentBase):
    """A ReAct agent implementation in AgentScope, which supports

    - Realtime steering
    - API-based (parallel) tool calling
    - Hooks around reasoning, acting, reply, observe and print functions
    - Structured output generation
    """

    finish_function_name: str = "generate_response"
    """The name of the function used to generate structured output. Only
    registered when structured output model is provided in the reply call."""

    def __init__(
        self,
        name: str,
        sys_prompt: str,
        model: ChatModelBase,
        formatter: FormatterBase,
        toolkit: Toolkit | None = None,
        parallel_tool_calls: bool = False,
        memory: MemoryBase | None = None,
        max_iters: int = 10, 
        print_hint_msg: bool = False       
            
    ) -> None:

        super().__init__()

        self.name = name
        self._sys_prompt = sys_prompt
        self.max_iters = max_iters
        self.model = model
        self.formatter = formatter
        # -------------- Memory management --------------
        # Record the dialogue history in the memory
        self.memory = memory or InMemoryMemory()

        self.intermediate_memory = []

        # -------------- Tool management --------------
        # If None, a default Toolkit will be created
        self.toolkit = toolkit or Toolkit()

        self.parallel_tool_calls = parallel_tool_calls

        # If print the reasoning hint messages
        self.print_hint_msg = print_hint_msg
        
        # The maximum number of iterations of the reasoning-acting loops
        self.max_iters = max_iters

        # The hint messages that will be attached to the prompt to guide the
        # agent's behavior before each reasoning step, and cleared after
        # each reasoning step, meaning the hint messages is one-time use only.
        # We use an InMemoryMemory instance to store the hint messages
        self._reasoning_hint_msgs = InMemoryMemory()


        # Variables to record the intermediate state
        # If required structured output model is provided
        self._required_structured_model: Type[BaseModel] | None = None


        # -------------- State registration and hooks --------------
        # Register the status variables
        self.register_state("name")
        self.register_state("_sys_prompt")
        


    @property
    def sys_prompt(self) -> str:
        return self._sys_prompt
    def generate_response(
        self,
        **kwargs: Any,
    ) -> ToolResponse:
        """
        Generate required structured output by this function and return it
        """

        structured_output = None
        # Prepare structured output
        if self._required_structured_model:
            try:
                # Use the metadata field of the message to store the
                # structured output
                structured_output = (
                    self._required_structured_model.model_validate(
                        kwargs,
                    ).model_dump()
                )

            except ValidationError as e:
                return ToolResponse(
                    content=[
                        TextBlock(
                            type="text",
                            text=f"Arguments Validation Error: {e}",
                        ),
                    ],
                    metadata={
                        "success": False,
                        "structured_output": {},
                    },
                )
        else:
            logger.warning(
                "The generate_response function is called when no structured "
                "output model is required.",
            )

        return ToolResponse(
            content=[
                TextBlock(
                    type="text",
                    text="Successfully generated response.",
                ),
            ],
            metadata={
                "success": True,
                "structured_output": structured_output,
            },
            is_last=True,
        )
    
    @trace_reply
    async def reply(  # pylint: disable=too-many-branches
        self,
        msg: Msg | list[Msg] | None = None,
        structured_model: Type[BaseModel] | None = None,
    ) -> Msg:
        """Generate a reply based on the current state and input arguments.

        Args:
            msg (`Msg | list[Msg] | None`, optional):
                The input message(s) to the agent.
            structured_model (`Type[BaseModel] | None`, optional):
                The required structured output model. If provided, the agent
                is expected to generate structured output in the `metadata`
                field of the output message.

        Returns:
            `Msg`:
                The output message generated by the agent.
        """
        # Record the input message(s) in the memory
        await self.memory.add(msg)

        # Control if LLM generates tool calls in each reasoning step
        tool_choice: Literal["auto", "none", "required"] | None = None
        
        # -------------- Structured output management --------------
        self._required_structured_model = structured_model
        # Record structured output model if provided
        if structured_model:
            # Register generate_response tool only when structured output
            # is required
            if self.finish_function_name not in self.toolkit.tools:
                self.toolkit.register_tool_function(
                    getattr(self, self.finish_function_name),
                )

            # Set the structured output model
            self.toolkit.set_extended_model(
                self.finish_function_name,
                structured_model,
            )
            tool_choice = "required"
        else:
            # Remove generate_response tool if no structured output is required
            self.toolkit.remove_tool_function(self.finish_function_name)